In [164]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate

#constants for 11B nuclei
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)

wkhz = w0*10**3 #Larmor freq in kHz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

# Coefficient for LHQ (cluster 1) from ASICS
A_coeff = [-1.730215,-2.744504,-3.396561]
B_coeff = [1.251296,2.561384,-1.133846]
C_coeff = [3.573296,0.986618,-0.473004]
D_coeff = [-0.060956,-0.376258,-0.980912]
E_coeff = [-0.037878,-0.579924,-1.166911]

# #Coefficient for LHQ (cluster 2) from ASICS
# A = [-2.237,-2.631,-3.326]
# B = [1.436,2.462,-1.178]
# C = [-3.527,0.272,-0.705]
# D = [0.223,-0.402,-0.926]
# E = [0.005,-0.526,-1.129]

In [165]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-35.1208684028640, 35.1208684028640], Ayz: [-61.5306552120510, 61.5306552120510]
Azz - Axx: [-189.595156285395, 189.595156285395], Axz: [-174.507296397013, 174.507296397013]
Ayy - Axx: [-249.031671571717, 249.031671571717], Axy: [-267.333199332134, 267.333199332134]


In [166]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print(combinations)

[(-189.595156285395, -249.031671571717, -35.1208684028640), (-189.595156285395, -249.031671571717, 35.1208684028640), (-189.595156285395, 249.031671571717, -35.1208684028640), (-189.595156285395, 249.031671571717, 35.1208684028640), (189.595156285395, -249.031671571717, -35.1208684028640), (189.595156285395, -249.031671571717, 35.1208684028640), (189.595156285395, 249.031671571717, -35.1208684028640), (189.595156285395, 249.031671571717, 35.1208684028640)]


In [167]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])
print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)


Axx1: [146.208942619037, 146.208942619037, -19.8121717621073, -19.8121717621073, 19.8121717621073, 19.8121717621073, -146.208942619037, -146.208942619037]
Axx2: [177.728070515432, 154.314158246856, -154.314158246856, -177.728070515432, 177.728070515432, 154.314158246856, -154.314158246856, -177.728070515432]
Axx3: [114.689814722642, 138.103726991218, 114.689814722642, 138.103726991218, -138.103726991218, -114.689814722642, -138.103726991218, -114.689814722642]
Ayy1: [-102.822728952680, -102.822728952680, 229.219499809609, 229.219499809609, -229.219499809609, -229.219499809609, 102.822728952680, 102.822728952680]
Ayy2: [-71.3036010562842, -94.7175133248602, 94.7175133248602, 71.3036010562842, -71.3036010562842, -94.7175133248602, 94.7175133248602, 71.3036010562842]
Ayy3: [-39.7844731598888, -86.6122976970408, -39.7844731598888, -86.6122976970408, 86.6122976970408, 39.7844731598888, 86.6122976970408, 39.7844731598888]
Azz1: [-43.3862136663575, -43.3862136663575, -209.407328047502, -209.4

In [168]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*wkhz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*wkhz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*wkhz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)


[{Ayy_s: 6.86289619187512e-6, Azz_s: 4.20545917436024e-6, Ayz_s: -7.77914061205763e-6}, {Axx_s: 1.32888018076280e-5, Azz_s: 4.24975992636871e-6, Axz_s: -1.01200515561466e-5}, {Axx_s: 1.25634334089429e-5, Ayy_s: 8.01243877630728e-6, Axy_s: -5.89842130865167e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 1.29261176082855e-5 7.43766748409120e-6 4.22760955036447e-6 -7.77914061205763e-6 -1.01200515561466e-5 -5.89842130865167e-6


In [169]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;

print('Quadrupolar tensor (tenon frame): \n', Q_T)

#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T)



Quadrupolar tensor (tenon frame): 
 [[ 146.20894262 -267.33319933 -174.5072964 ]
 [-267.33319933  -94.71751332   61.53065521]
 [-174.5072964    61.53065521  -51.49142929]]
Chemical Shift tensor (tenon frame): 
 [[ 1.29261176e-05 -5.89842131e-06 -1.01200516e-05]
 [-5.89842131e-06  7.43766748e-06 -7.77914061e-06]
 [-1.01200516e-05 -7.77914061e-06  4.22760955e-06]]


In [170]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS
eigenvalues, eigenvectors = np.linalg.eig(Q_T)
D_quad = np.diag(eigenvalues)
print('Diagonalized Quadrupolar Tensor:\n', D_quad, '\n')

quad_avg = np.mean(eigenvalues) # Tr(A)Quad/3
sorted_eigenvalues = sorted((eigenvalues - quad_avg), key=abs)

print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues, '\n') 

Vyy = (sorted_eigenvalues[0]+ quad_avg)*(2*Ispin*(2*Ispin - 1)) 
Vxx = (sorted_eigenvalues[1] + quad_avg)*(2*Ispin*(2*Ispin - 1))
Vzz = (sorted_eigenvalues[2] + quad_avg)*(2*Ispin*(2*Ispin - 1)) 

print('Quad Tensors Vzz, Vyy, Vxx: \n', Vzz, Vyy, Vxx)

print('================================================================================================')
#Calculate Quadrupolar Tensor in PAS
eigenvalues, eigenvectors = np.linalg.eig(CS_T)
D_cs = np.diag(eigenvalues)
print('Diagonalized CSA Tensor:\n', D_cs, '\n')

cs_avg = np.mean(eigenvalues) # Tr(A)CSA/3
sorted_eigenvalues = sorted((eigenvalues - cs_avg), key=abs)

print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues, '\n') 

csyy = -(sorted_eigenvalues[0] + cs_avg) 
csxx = -(sorted_eigenvalues[1] + cs_avg)
cszz = -(sorted_eigenvalues[2] + cs_avg) 

print('Quad Tensors δzz, δyy, δxx: \n', cszz, csyy, csxx)

Diagonalized Quadrupolar Tensor:
 [[ 392.39282568    0.            0.        ]
 [   0.         -278.21272231    0.        ]
 [   0.            0.         -114.18010337]] 

Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):
 [-114.18010336596872, -278.2127223135424, 392.39282567951113] 

Quad Tensors Vzz, Vyy, Vxx: 
 2354.3569540770677 -685.0806201958115 -1669.2763338812535
Diagonalized CSA Tensor:
 [[-8.42548976e-06  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.96448082e-05  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.33720762e-05]] 

Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):
 [5.174944666045168e-06, 1.1447676641225647e-05, -1.6622621307270814e-05] 

Quad Tensors δzz, δyy, δxx: 
 8.425489759690434e-06 -1.3372076213625548e-05 -1.9644808188806026e-05


In [171]:
#Quadrupolar tensor parameters
cq = Vzz/10**3
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
iso_cs = np.mean([cszz, csyy, csxx]) #converting kHz to ppm (1 kHz = 10**3/Larmor frequency) #Issues with conversion?
csa = cszz - iso_cs
etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs], ['csa (ppm)', csa], ['etas', etas] ]
print(tabulate(table, headers=['Quantity', 'Fit Value']))

Quantity         Fit Value
------------  ------------
cq (MHz)       2.35436
etaq           0.418032
iso_cs (ppm)  -8.19713e-06
csa (ppm)      1.66226e-05
etas           0.377361
